# 🎯 [C202_M10] 대역폭 제한 환경 메시지 큐·재전송 시스템 통합 및 전술통신 브리핑 패키지 완성
### Google Colab 실습 환경 (지휘통제(C2) / Level 2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C202/c202_m10_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 2) |
| **미션 ID** | C202_M10 |
| **미션 주제** | 대역폭 제한 환경 메시지 큐·재전송 시스템 통합 및 전술통신 브리핑 패키지 완성 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 전자전 재밍과 통신 장비 부족으로 협대역 무선망의 대역폭이 크게 줄어드는 상황이 잦았고, 이때 적 접촉 보고나 의무후송 요청 같은 긴급 전문이 행정 전문 뒤에 묶여 늦게 전달되는 문제가 지휘통제의 약점이 되었다. 본 프로젝트에서는 가상 협대역 링크(2.4kbps)와 가상 전술 전문을 코드로 직접 만들어, 전문 구조 정의, 패킷 파싱·압축 전처리, 긴급도 기반 우선순위 큐, 채널 상태에 따른 지능형 재전송, 링크 신뢰도 산출, 통신 관제 화면, 통신관 HITL 통제, 대역폭 급감 스트레스 시험, AAR 재조정, 최종 통합까지 메시지 우선순위 큐·재전송 프로토타입을 단계별로 구현한다. 최종 목표는 대역폭이 제한된 상황에서도 FLASH·IMMEDIATE 전문을 기한 안에 전달하고, 어떤 전문도 잃어버리지 않으며, 순서 조정의 최종 판단은 통신관이 내리도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c202_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c202.py` | 전처리, 우선순위 큐, 재전송 제어, 링크 신뢰도, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c202_data_spec.md`, `docs/c202_hitl_guide.md`, `docs/c202_aar_report.md`, `docs/c202_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c202_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c202_stress_test.py` | 대역폭 90% 급감 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c202_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 10. 대역폭 제한 환경 메시지 큐·재전송 시스템 통합 및 전술통신 브리핑 패키지 완성
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

대역폭 제한 시 메시지 우선순위 큐·재전송 시스템 통합 및 배포 가이드를 완성한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **패킷 수신부터 전처리, 큐·재전송, 링크 등급, 통신관 요청 대기열, 관제 화면까지 통합하고 전술통신 브리핑 패키지를 완성한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **재조정 파라미터로 전체 과정을 한 번에 실행하는 통합 코드를 완성하고 손상·미등록 패킷이 격리되는지 확인하고 유효성을 점검하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 2개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/main_c202.py, README.md, docs/c202_final_report.pdf(md)` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 군 실무 즉시 도입 가능성, 운용 신뢰도 및 전략적 기대효과

#### 미션 PASS 조건

- 정량 기준: End-to-End 통합 파이프라인 실행 성공률 100%, 테스트 커버리지 85% 이상


- 진행 기준: 원클릭 실행 스크립트 동작 확인 및 최종 발표 산출물 작성 완료


- 검증 방법: 최종 검증 위원회(평가관) 모의 시연 및 전체 기능 통합 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""원시 전문 패킷 파싱·검증·압축부터 우선순위 큐, 지능형 재전송, 링크 신뢰도 등급, 통신관 HITL 요청 대기열, 관제 화면, AAR 재보정 파라미터 적용까지 전체 파이프라인을 한 파일로 통합 실행하고, 함수 커버리지를 포함한 자체 점검 결과와 전술통신 브리핑 요약을 출력한다."""
import hashlib
import inspect
import json
import sys
import time
import zlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 전술 전문(메시지) 생성기 (실제 부대·통신망·제원 정보 없음)
# 긴급도 4단계(FLASH/IMMEDIATE/PRIORITY/ROUTINE)와 전문 유형별 크기를 가진 전문을 만든다.
# ---------------------------------------------------------------------------
PREC = ["FLASH", "IMMEDIATE", "PRIORITY", "ROUTINE"]
RANK = {p: i for i, p in enumerate(PREC)}
DEADLINE = {"FLASH": 10, "IMMEDIATE": 30, "PRIORITY": 120, "ROUTINE": 600}   # 가상 전달 목표 시간(초)
MSG_TYPES = {   # 유형: (긴급도 후보, 확률, 본문 크기 범위 bytes, 발생 비율)
    "CONTACT": (["FLASH", "IMMEDIATE"], [0.7, 0.3], (120, 300), 0.08),
    "MEDEVAC": (["FLASH", "IMMEDIATE"], [0.4, 0.6], (150, 350), 0.05),
    "SITREP": (["IMMEDIATE", "PRIORITY"], [0.2, 0.8], (600, 2500), 0.20),
    "POSREP": (["PRIORITY", "ROUTINE"], [0.5, 0.5], (60, 120), 0.35),
    "LOGREQ": (["PRIORITY", "ROUTINE"], [0.4, 0.6], (300, 1200), 0.14),
    "ADMIN": (["ROUTINE"], [1.0], (1000, 5000), 0.18),
}


def make_messages(duration=600, rate=0.12, seed=7, t_start=0.0, prefix="M"):
    """duration초 동안 평균 rate건/초로 발생하는 가상 전문 목록."""
    rng = np.random.default_rng(seed)
    n = int(rng.poisson(rate * duration))
    times = np.sort(rng.uniform(t_start, t_start + duration, n))
    types = list(MSG_TYPES)
    msgs = []
    for i, t0 in enumerate(times):
        ty = types[rng.choice(len(types), p=[MSG_TYPES[k][3] for k in types])]
        cands, p, (lo, hi), _ = MSG_TYPES[ty]
        msgs.append({"id": f"{prefix}{i:04d}", "t0": float(t0), "type": ty, "prec": str(rng.choice(cands, p=p)),
                     "size": int(rng.integers(lo, hi + 1)), "src": f"U{int(rng.integers(1, 9)):02d}"})
    return msgs


class Link:
    """[교육용가상자료] 가상 협대역 무선 링크(기본 2.4kbps). 초 단위 상태(good/bad/jam)에 따라
    비트오류율이 바뀌고(Gilbert-Elliott 모델), 구간별 대역폭 감소도 줄 수 있다."""
    BER = {"good": 1e-5, "bad": 2e-4, "jam": 1.2e-3}

    def __init__(self, horizon=4000, bw_bps=2400, seed=1, p_gb=0.02, p_bg=0.15, jam=(), bw_drop=(), rtt=(1.2, 0.4)):
        rng = np.random.default_rng(seed)
        st, s = [], "good"
        for _ in range(int(horizon) + 1):
            s = ("bad" if rng.random() < p_gb else "good") if s == "good" else ("good" if rng.random() < p_bg else "bad")
            st.append(s)
        for a, b in jam:
            for k in range(int(a), min(int(b), len(st))):
                st[k] = "jam"
        self.states, self.bw_bps, self.bw_drop, self.rtt_par = st, bw_bps, list(bw_drop), rtt

    def state(self, t):
        return self.states[min(int(t), len(self.states) - 1)]

    def bw(self, t):
        f = 1.0
        for a, b, fac in self.bw_drop:
            if a <= t < b:
                f = fac
        return self.bw_bps * f

    def p_loss(self, nbytes, t):
        return 1 - (1 - self.BER[self.state(t)]) ** (8 * nbytes)

    def rtt(self, rng):
        return self.rtt_par[0] + rng.exponential(self.rtt_par[1])
# ---------------------------------------------------------------------------

# ---- 우선순위 큐·재전송 이산 사건 시뮬레이터 공통 함수 ----
HDR = 32   # 전문 조각(패킷) 헤더 bytes (미션 2의 헤더 압축 적용 시 12 bytes)


def policy_key(m, t, policy, aging):
    if policy == "fifo":                                   # 도착 순서
        return (m["t0"],)
    if policy == "strict":                                 # 긴급도 우선, 같으면 도착 순서
        return (RANK[m["prec"]], m["t0"])
    # aging: 긴급도 + (대기시간 / 전달 목표시간) 가중. 오래 기다린 전문과 기한이 임박한 전문이 앞으로 온다.
    return (RANK[m["prec"]] - aging * (t - m["t0"]) / DEADLINE[m["prec"]], m["t0"])


def simulate(msgs, link, policy="strict", mtu=256, preempt=True, rto_mode="fixed", rto_s=4.0, aging=0.0,
             redundancy=False, adapt_mtu=False, hdr=HDR, seed=0, until=None, admit=None):
    """전문 목록을 링크로 보내는 이산 사건 시뮬레이션.
    - 전문은 mtu 크기 조각으로 나눠 보낸다. 조각이 손실되면 RTO 후 재전송한다(자료는 버리지 않음).
    - preempt=False면 한 전문의 조각을 끝까지 연속 전송한다(큰 전문이 긴급 전문을 막는 현상 재현).
    - rto_mode='adaptive'면 왕복시간(RTT) 추정값으로 RTO를 계산한다(Jacobson 방식).
    - redundancy=True면 링크 열화 추정 시 FLASH·IMMEDIATE 조각을 2회 연속 전송한다.
    - adapt_mtu=True면 링크 열화 추정 시 조각 크기를 줄인다.
    - admit(m, t)이 False인 전문은 새 조각을 보내지 않고 저장해 둔다(성능 저하 모드)."""
    rng = np.random.default_rng(seed)
    pend = sorted(({**m, "unsent": m["size"], "ready": [], "frags": [], "done": None, "first": None} for m in msgs), key=lambda m: m["t0"])
    t, i, active, cur = 0.0, 0, [], None
    srtt, rttvar, loss = None, None, 0.0
    st = {"tx": 0, "retx": 0, "spurious": 0, "bytes": 0, "decide_ms": [], "flag": {}, "log": []}
    until = until if until is not None else (pend[-1]["t0"] if pend else 0) + 6000
    while (i < len(pend) or active) and t < until:
        while i < len(pend) and pend[i]["t0"] <= t:
            active.append(pend[i]); i += 1
        td = time.perf_counter()
        for m in active:                                   # ACK가 이미 도착한 재전송 예약은 취소
            m["ready"] = [e for e in m["ready"] if not (e[1]["ack"] is not None and e[1]["ack"] <= t)]
        ready = [m for m in active if any(r <= t for r, _ in m["ready"]) or (m["unsent"] > 0 and (admit is None or admit(m, t)))]
        if not ready:
            nxt = ([pend[i]["t0"]] if i < len(pend) else []) + [r for m in active for r, _ in m["ready"] if r > t]
            t = min(nxt) if nxt else t + 1.0
            continue
        if not preempt and cur in ready and cur["unsent"] > 0:
            m = cur
        else:
            m = min(ready, key=lambda x: policy_key(x, t, policy, aging))
        st["decide_ms"].append((time.perf_counter() - td) * 1000)
        due = [e for e in m["ready"] if e[0] <= t]
        if due:
            e = min(due, key=lambda x: x[0]); m["ready"].remove(e); frag = e[1]
            st["retx"] += 1; st["spurious"] += frag["recv"]
        else:
            size = mtu if not adapt_mtu else (mtu if loss < 0.15 else (mtu // 2 if loss < 0.4 else mtu // 4))
            size = min(m["unsent"], size); m["unsent"] -= size
            frag = {"b": size, "recv": False, "ack": None, "tries": 0}
            m["frags"].append(frag)
        cur = m
        m["first"] = t if m["first"] is None else m["first"]
        nb = frag["b"] + hdr
        copies = 2 if (redundancy and RANK[m["prec"]] <= 1 and loss > 0.15) else 1
        for _ in range(copies):
            t += nb * 8 / link.bw(t)
            st["tx"] += 1; st["bytes"] += nb
            ok = rng.random() >= link.p_loss(nb, t)
            st["log"].append((t, nb, ok, m["prec"], m["id"]))            # 전송 기록(시각, bytes, 성공, 긴급도, 전문)
            loss = 0.85 * loss + 0.15 * (0.0 if ok else 1.0)          # 최근 손실률 추정(EWMA)
            st["flag"][int(t)] = st["flag"].get(int(t), False) or loss > 0.12   # 링크 열화 판정(초 단위)
            if ok and not frag["recv"]:
                frag["recv"], frag["rt"] = True, t
                r = link.rtt(rng); frag["ack"] = t + r
                if srtt is None:
                    srtt, rttvar = r, r / 2
                else:
                    rttvar = 0.75 * rttvar + 0.25 * abs(srtt - r); srtt = 0.875 * srtt + 0.125 * r
        frag["tries"] += 1
        rto = rto_s if (rto_mode == "fixed" or srtt is None) else float(np.clip(srtt + 4 * rttvar, 1.0, 30.0))
        if not frag["recv"] or frag["ack"] > t + rto:                   # 손실 또는 ACK 지연 -> 재전송 예약
            m["ready"].append((t + rto, frag))
        if m["done"] is None and m["unsent"] == 0 and all(f["recv"] for f in m["frags"]):
            m["done"] = max(f["rt"] for f in m["frags"])
        active = [x for x in active if not (x["unsent"] == 0 and not x["ready"] and all(f["recv"] for f in x["frags"]))]
    res = [{"id": m["id"], "type": m["type"], "prec": m["prec"], "size": m["size"], "t0": m["t0"], "done": m["done"], "wait": None if m["first"] is None else m["first"] - m["t0"],
            "retx": sum(f["tries"] for f in m["frags"]) - len(m["frags"]),
            "latency": None if m["done"] is None else m["done"] - m["t0"]} for m in pend]
    for r in res:
        r["on_time"] = r["latency"] is not None and r["latency"] <= DEADLINE[r["prec"]]
    return res, st


def summarize(res):
    """긴급도별 기한 내 전달률·지연. 종합 정확도 = 긴급도별 기한 내 전달률의 평균."""
    df = pd.DataFrame(res)
    rows = []
    for p in PREC:
        d = df[df["prec"] == p]
        lat = d["latency"].dropna()
        rows.append({"긴급도": p, "건수": len(d), "기한내 전달률": round(d["on_time"].mean(), 3) if len(d) else np.nan,
                     "평균 지연(s)": round(lat.mean(), 1) if len(lat) else np.nan,
                     "95% 지연(s)": round(lat.quantile(0.95), 1) if len(lat) else np.nan, "미전달": int(d["done"].isna().sum())})
    tab = pd.DataFrame(rows)
    return tab, round(float(tab["기한내 전달률"].mean()), 3)

CONFIG = {"policy": "strict", "preempt": True, "rto_mode": "fixed", "rto_s": 6.0, "mtu": 128, "aging": 0.0,   # 미션 9 재보정값
          "redundancy": True, "hdr": 14}                                                                 # 미션 4 중복 전송, 미션 2 헤더 압축
ALIAS = {"Z": "FLASH", "O": "IMMEDIATE", "P": "PRIORITY", "R": "ROUTINE"}
ROSTER = {f"U{i:02d}" for i in range(1, 9)}


def to_packet(m):
    body = "".join(f"{m['type']} {m['src']} I{(k * 7919 + int(m['t0'] * 10)) % 997:03d} Q{(k * 31 + m['size']) % 89:02d};" for k in range(m["size"] // 20 + 1))[:m["size"]]
    return f"ZCZC|ID={m['id']}|PREC={'ZOPR'[RANK[m['prec']]]}|TYPE={m['type']}|SRC={m['src']}|T={m['t0']:.1f}|CRC={zlib.crc32(body.encode()):08x}|BODY={body}"


def parse(line):
    kv = dict(p.split("=", 1) for p in line.split("|")[1:] if "=" in p)
    if set(kv) != {"ID", "PREC", "TYPE", "SRC", "T", "CRC", "BODY"} or kv["PREC"] not in ALIAS or kv["SRC"] not in ROSTER:
        raise ValueError("형식·긴급도·송신국 오류")
    if f"{zlib.crc32(kv['BODY'].encode()):08x}" != kv["CRC"]:
        raise ValueError("CRC 불일치")
    body_z = zlib.compress(kv["BODY"].encode(), 9)
    return {"id": kv["ID"], "prec": ALIAS[kv["PREC"]], "type": kv["TYPE"], "src": kv["SRC"], "t0": float(kv["T"]), "size": len(body_z)}


def link_grade(loss):
    return "High" if loss < 0.1 else ("Medium" if loss < 0.3 else "Low")


class RequestQueue:
    """통신관 조작 요청 대기열: 시스템은 요청만 만들고 적용은 사람이 한다."""

    def __init__(self):
        self.items, self.log = [], []

    def submit(self, kind, target, why):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"event": "REQUEST", "kind": kind, "target": target, "why": why}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, ensure_ascii=False, sort_keys=True)).encode()).hexdigest()})
        self.items.append({**body, "status": "PENDING"})


# 함수 커버리지 측정(표준 라이브러리 sys.setprofile): 이 파일에서 정의한 함수 중 실제 호출된 비율
called = set()
sys.setprofile(lambda f, e, a: called.add(f.f_code.co_name) if e == "call" and f.f_globals is globals() else None)
t_start = time.perf_counter()

# ---------------- 1. 수신·전처리 ----------------
msgs = make_messages(1200, 0.5, seed=21)          # 평시의 약 2.4배 트래픽
lines = [to_packet(m) for m in msgs]
lines[5] = lines[5].replace("SRC=U", "SRC=X")                # 손상·미등록 송신국 패킷 주입
lines[9] = lines[9][:-3] + "###"
parsed, rejected = [], {}
for ln in lines:
    try:
        parsed.append(parse(ln))
    except ValueError as e:
        rejected[ln.split("|")[1][3:]] = str(e)

# ---------------- 2. 큐·재전송 ----------------
link = Link(12000, seed=31, jam=[(300, 360)], bw_drop=[(600, 800, 0.5)], rtt=(2.0, 0.8))
res, st = simulate(parsed, link, **CONFIG)
tab, acc = summarize(res)
lg = pd.DataFrame(st["log"], columns=["t", "bytes", "ok", "prec", "id"])
lg["sec"] = (lg["t"] // 10 * 10).astype(int)
win = lg.groupby("sec")["ok"].apply(lambda s: 1 - s.mean()).rename("loss").reset_index()
win["grade"] = win["loss"].map(link_grade)

# ---------------- 3. HITL 요청(자동 적용 없음) ----------------
rq = RequestQueue()
for _, w in win[win["grade"] == "Low"].iterrows():
    rq.submit("ROUTE_REVIEW", f"t={w['sec']}s", f"손실률 {w['loss']:.0%}, 경로·중계 검토")
for r in res:
    if r["prec"] == "FLASH" and not r["on_time"]:
        rq.submit("FLASH_LATE", r["id"], f"지연 {r['latency']:.1f}s, 통신관 확인")
sys.setprofile(None)

# ---------------- 4. 자체 점검 ----------------
mine = [n for n, o in list(globals().items()) if inspect.isfunction(o) and o.__module__ == "__main__"]
mine += ["submit", "state", "bw", "p_loss", "rtt", "__init__"]
cov = len(set(mine) & called) / len(set(mine))
urg = np.mean([r["on_time"] for r in res if RANK[r["prec"]] <= 1])
elapsed = time.perf_counter() - t_start
chk = {
    "손상 패킷 2건 격리": len(rejected) == 2,
    "정상 전문 전부 큐 등록": len(parsed) == len(msgs) - 2,
    "전문 유실 0건(전달+보관=전체)": sum(r["done"] is not None for r in res) + sum(r["done"] is None for r in res) == len(parsed),
    "FLASH 기한 내 전달률 90% 이상": tab.loc[0, "기한내 전달률"] >= 0.9,
    "긴급 전문 기한 내 85% 이상": urg >= 0.85,
    "종합 정확도 0.9 이상": acc >= 0.9,
    "자동 적용 0건(요청 전부 PENDING)": all(i["status"] == "PENDING" for i in rq.items),
    "감사 로그 건수 일치": len(rq.log) == len(rq.items),
    "스케줄링 결정 200ms 이내": max(st["decide_ms"]) < 200,
    "함수 커버리지 85% 이상": cov >= 0.85,
}
passed = sum(chk.values())
for k, v in chk.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"자체 점검 {passed}/{len(chk)} 통과, 함수 커버리지 {cov:.0%}, 파이프라인 실행 성공 {'예' if passed == len(chk) else '아니오'}")
print(tab[["긴급도", "건수", "기한내 전달률", "95% 지연(s)", "미전달"]].to_string(index=False))

# ---------------- 5. 관제 화면 ----------------
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0, 0].bar(tab["긴급도"], tab["기한내 전달률"], color=["red", "orange", "deepskyblue", "gray"]); ax[0, 0].set_ylim(0, 1.05); ax[0, 0].set_title("On-time ratio by precedence")
ax[0, 1].plot(win["sec"], win["loss"] * 100); ax[0, 1].axhline(30, c="r", ls="--"); ax[0, 1].set_title("Loss rate per 10 s (%)")
for a, b, c in [(300, 360, "purple"), (600, 800, "gold")]:
    ax[0, 1].axvspan(a, b, color=c, alpha=0.2)
lat = pd.DataFrame(res)
for p, c in zip(PREC, ["red", "orange", "deepskyblue", "gray"]):
    d = lat[lat["prec"] == p]; ax[1, 0].scatter(d["t0"], d["latency"], s=8, c=c, label=p)
ax[1, 0].set_yscale("log"); ax[1, 0].legend(fontsize=7); ax[1, 0].set_title("Delivery latency (s, log)")
ax[1, 1].axis("off")
ax[1, 1].text(0, 0.9, f"Messages     {len(parsed)} (+{len(rejected)} rejected)\nOverall acc  {acc:.3f}\nUrgent ok    {urg:.1%}\n"
              f"HITL pending {len(rq.items)}\nSelf-check   {passed}/{len(chk)}", family="monospace", fontsize=13, va="top")
plt.suptitle("C202 integrated message queue - training data only"); plt.tight_layout(); plt.show()

# ---------------- 6. 전술통신 브리핑 요약 ----------------
print(f"""
[전술통신 브리핑 요약 - 교육용가상자료]
1. 처리: 수신 {len(lines)}건 중 {len(parsed)}건 전송, 손상·미등록 {len(rejected)}건 격리, 전체 소요 {elapsed:.1f}s
2. 성능: 종합 정확도 {acc:.3f}, FLASH 기한 내 {tab.loc[0, '기한내 전달률']:.1%}, 긴급 전문 기한 내 {urg:.1%}, 재전송 {st['retx']}회
3. 통신관 검토 요청: {len(rq.items)}건(경로 검토 {sum(i['kind'] == 'ROUTE_REVIEW' for i in rq.items)}, FLASH 지연 {sum(i['kind'] == 'FLASH_LATE' for i in rq.items)}), 자동 적용 0건
4. 운용 유의: 강제 전송·보류는 통신관이 근거를 기록해 수행하고, FLASH보다 앞서는 조작은 상급자 승인을 받는다.
5. 한계·확장: 가상 링크 모델 기준 결과로 실제 장비·망에서는 재측정이 필요하다. 인접 체계 연동은 전문 입력·전달 결과 조회 API로 확장한다.""")


<details>
<summary><b>💡 [참고] 미션 10 예상 실행 결과 (Output)</b></summary>

```text
[PASS] 손상 패킷 2건 격리
[PASS] 정상 전문 전부 큐 등록
[PASS] 전문 유실 0건(전달+보관=전체)
[PASS] FLASH 기한 내 전달률 90% 이상
[PASS] 긴급 전문 기한 내 85% 이상
[PASS] 종합 정확도 0.9 이상
[PASS] 자동 적용 0건(요청 전부 PENDING)
[PASS] 감사 로그 건수 일치
[PASS] 스케줄링 결정 200ms 이내
[PASS] 함수 커버리지 85% 이상
자체 점검 10/10 통과, 함수 커버리지 100%, 파이프라인 실행 성공 예
      긴급도  건수  기한내 전달률  95% 지연(s)  미전달
    FLASH  45    0.978        7.3    0
IMMEDIATE  55    1.000        7.3    0
 PRIORITY 248    1.000        8.3    0
  ROUTINE 271    1.000       46.8    0

[전술통신 브리핑 요약 - 교육용가상자료]
1. 처리: 수신 621건 중 619건 전송, 손상·미등록 2건 격리, 전체 소요 0.2s
2. 성능: 종합 정확도 0.994, FLASH 기한 내 97.8%, 긴급 전문 기한 내 99.0%, 재전송 111회
3. 통신관 검토 요청: 7건(경로 검토 6, FLASH 지연 1), 자동 적용 0건
4. 운용 유의: 강제 전송·보류는 통신관이 근거를 기록해 수행하고, FLASH보다 앞서는 조작은 상급자 승인을 받는다.
5. 한계·확장: 가상 링크 모델 기준 결과로 실제 장비·망에서는 재측정이 필요하다. 인접 체계 연동은 전문 입력·전달 결과 조회 API로 확장한다.
```
</details>
